In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

import numpy as np
import time

DATA_PROCESSED = Path("../data/processed")

## 1. Load Graph Data

In [8]:
def load_graph(name):
    path = DATA_PROCESSED / f"{name}_final.pt"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found.")
    data = torch.load(path, weights_only=False)
    print(f"Loaded {name}")
    print(data)
    return data

dataset_name = "elliptic"

data = load_graph(dataset_name)

Loaded elliptic
Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], timesteps=[203769], num_nodes=203769, train_mask=[203769], test_mask=[203769], val_mask=[203769])


In [9]:
def evaluate_predictions(y_true, y_pred, y_prob=None):
    results = {}
    results["F1"] = f1_score(y_true, y_pred)
    results["Precision"] = precision_score(y_true, y_pred)
    results["Recall"] = recall_score(y_true, y_pred)
    
    if y_prob is not None:
        results["AUC"] = roc_auc_score(y_true, y_prob)
    
    return results

## 2. Pure Feature Models


### Logistic regression

In [10]:
# Prepare data
X = data.x.cpu().numpy()
y = data.y.cpu().numpy()

train_idx = data.train_mask.cpu().numpy()
test_idx = data.test_mask.cpu().numpy()

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train Logistic Regression
start = time.time()
lr_model = LogisticRegression(max_iter=500)
lr_model.fit(X_train, y_train)
train_time = time.time() - start

# Predict
y_pred = lr_model.predict(X_test)
y_prob = lr_model.predict_proba(X_test)[:, 1]

results_lr = evaluate_predictions(y_test, y_pred, y_prob)
results_lr["Runtime"] = train_time

results_lr

{'F1': 0.3098544420277731,
 'Precision': 0.18921127911728647,
 'Recall': 0.8550323176361958,
 'AUC': 0.8738900725863545,
 'Runtime': 1.2335875034332275}

### Simple MLP

In [11]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 2)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleMLP(data.x.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

data = data.to(device)

start = time.time()

for epoch in range(50):
    model.train()
    optimizer.zero_grad()
    out = model(data.x)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

train_time = time.time() - start

model.eval()
with torch.no_grad():
    logits = model(data.x)
    preds = logits[data.test_mask].argmax(dim=1)
    probs = F.softmax(logits[data.test_mask], dim=1)[:, 1]

results_mlp = evaluate_predictions(
    data.y[data.test_mask].cpu(),
    preds.cpu(),
    probs.cpu()
)

results_mlp["Runtime"] = train_time
results_mlp

{'F1': 0.3375980189847297,
 'Precision': 0.21737975019930905,
 'Recall': 0.7553093259464451,
 'AUC': 0.8586241073470737,
 'Runtime': 5.2058000564575195}

### Graph-Aware Linear Model


In [12]:
def one_hop_aggregate(data):
    src, dst = data.edge_index
    x = data.x
    
    num_nodes = data.num_nodes
    
    deg = torch.zeros(num_nodes, device=x.device)
    deg.scatter_add_(0, dst, torch.ones_like(dst, dtype=torch.float))
    deg[deg == 0] = 1
    
    aggregated = torch.zeros_like(x)
    aggregated.index_add_(0, dst, x[src])
    
    aggregated = aggregated / deg.unsqueeze(1)
    
    return aggregated

In [13]:
agg_features = one_hop_aggregate(data)

X_agg = agg_features.cpu().numpy()

X_train_agg = X_agg[train_idx]
X_test_agg = X_agg[test_idx]

scaler = StandardScaler()
X_train_agg = scaler.fit_transform(X_train_agg)
X_test_agg = scaler.transform(X_test_agg)

start = time.time()
lr_graph = LogisticRegression(max_iter=500)
lr_graph.fit(X_train_agg, y_train)
train_time = time.time() - start

y_pred = lr_graph.predict(X_test_agg)
y_prob = lr_graph.predict_proba(X_test_agg)[:, 1]

results_graph_linear = evaluate_predictions(y_test, y_pred, y_prob)
results_graph_linear["Runtime"] = train_time

results_graph_linear

{'F1': 0.25793357933579336,
 'Precision': 0.1611713165782799,
 'Recall': 0.6454293628808865,
 'AUC': 0.782096866597108,
 'Runtime': 0.9572038650512695}

### Basic GCN 

In [14]:
class BasicGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, num_nodes):
        src, dst = edge_index
        
        deg = torch.zeros(num_nodes, device=x.device)
        deg.scatter_add_(0, dst, torch.ones_like(dst, dtype=torch.float))
        deg[deg == 0] = 1
        
        agg = torch.zeros_like(x)
        agg.index_add_(0, dst, x[src])
        agg = agg / deg.unsqueeze(1)
        
        return self.linear(agg)

In [15]:
class BasicGCN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.gcn1 = BasicGCNLayer(input_dim, hidden_dim)
        self.gcn2 = BasicGCNLayer(hidden_dim, 2)

    def forward(self, data):
        x = self.gcn1(data.x, data.edge_index, data.num_nodes)
        x = F.relu(x)
        x = self.gcn2(x, data.edge_index, data.num_nodes)
        return x

In [16]:
model = BasicGCN(data.x.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

start = time.time()

for epoch in range(50):
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

train_time = time.time() - start

model.eval()
with torch.no_grad():
    logits = model(data)
    preds = logits[data.test_mask].argmax(dim=1)
    probs = F.softmax(logits[data.test_mask], dim=1)[:, 1]

results_gcn = evaluate_predictions(
    data.y[data.test_mask].cpu(),
    preds.cpu(),
    probs.cpu()
)

results_gcn["Runtime"] = train_time
results_gcn

{'F1': 0.15811801002699577,
 'Precision': 0.1357615894039735,
 'Recall': 0.18928901200369344,
 'AUC': 0.5862573050049225,
 'Runtime': 19.86916708946228}

## 3.Model Comparison


In [17]:
import pandas as pd

comparison = pd.DataFrame([
    {"Model": "Logistic Regression", **results_lr},
    {"Model": "MLP", **results_mlp},
    {"Model": "Graph Linear", **results_graph_linear},
    {"Model": "Basic GCN", **results_gcn}
])

comparison

,Model,F1,Precision,Recall,AUC,Runtime
0,Logistic Regression,0.309854,0.189211,0.855032,0.873890,1.233588
1,MLP,0.337598,0.217380,0.755309,0.858624,5.205800
2,Graph Linear,0.257934,0.161171,0.645429,0.782097,0.957204
3,Basic GCN,0.158118,0.135762,0.189289,0.586257,19.869167
